In [1]:
from sympy import *
from cq.symbolic import *

D = 2
rho = Matrix([[Symbol(f'rho{min(i, j)}{max(i, j)}') for i in range(D+1)] for j in range(D+1)])
rho

Matrix([
[rho00, rho01, rho02],
[rho01, rho11, rho12],
[rho02, rho12, rho22]])

In [2]:
T = T_matrix(D)
T

Matrix([
[       1/4,   0, -sqrt(2)/4],
[         0, 3/4,          0],
[-sqrt(2)/4,   0,        5/4]])

In [3]:
T = (rho@T).trace()
T

rho00/4 - sqrt(2)*rho02/2 + 3*rho11/4 + 5*rho22/4

In [4]:
g = rho_to_g(rho)
g

Matrix([
[                    rho00 + rho11 + rho22],
[                2*rho01 + 2*sqrt(2)*rho12],
[2*rho02 + sqrt(2)*rho11 + 2*sqrt(2)*rho22],
[                          2*sqrt(3)*rho12],
[                            sqrt(6)*rho22]])

## Vector to matrix

In [5]:
equations = []
for i, pi in enumerate(g):
    equations.append(pi - Symbol(f'g{i}'))
equations.append(T - Symbol('T'))
equations = Matrix(equations)
equations

Matrix([
[                           -g0 + rho00 + rho11 + rho22],
[                       -g1 + 2*rho01 + 2*sqrt(2)*rho12],
[       -g2 + 2*rho02 + sqrt(2)*rho11 + 2*sqrt(2)*rho22],
[                                 -g3 + 2*sqrt(3)*rho12],
[                                   -g4 + sqrt(6)*rho22],
[-T + rho00/4 - sqrt(2)*rho02/2 + 3*rho11/4 + 5*rho22/4]])

In [6]:
from sympy.abc import x, y, t
from IPython.display import display, Markdown

def linear_solve(p:Poly, *t) -> tuple[Expr, ...]:
    """Return parametrised linear solutions.
    
    Returns $x$ for $ax+b=0$
    and $x$, $y$ for ax+by+c=0.
    """
    assert isinstance(p, Poly)
    assert len(t) == len(p.gens)-1
    d = p.as_dict()
    
    match len(p.gens):
        case 1: #ax+b=0
            assert d.keys() <= {(0,), (1,)}
            return (p.root(0), )
        
        case 2: #ax+by+c=0
            assert d.keys() <= {(0,0), (0,1), (1,0)}
            
            a = d.get((1,0), Integer(0))
            b = d.get((0,1), Integer(0))
            c = d.get((0,0), Integer(0))
            x = (-a*c/(a**2+b**2)+b*t[0]).simplify()
            y = (-b*c/(a**2+b**2)-a*t[0]).simplify()
            return x, y
        
        case _:
            raise NotImplementedError("higher dimension not yet implemented")

for i in reversed(range(len(equations)-1)):
    print('Using')
    display(equations[i])
    
    s = set(rho) & equations[i].free_symbols
    print('to solve for')
    display(Matrix(tuple(s)))
    
    p = Poly(equations[i], *s)
    s_sol = linear_solve(p) if len(s)==1 else linear_solve(p, Symbol(f't{i}'))
    print('Solution:')
    display(Matrix(s_sol))
    
    subs = {si:s_soli for si, s_soli in zip(s, s_sol)}
    
    rho = rho.subs(subs).expand()
    equations = equations.subs(subs).expand()
    print('Resulting in')
    display(rho)
    display(equations)
    display(Markdown('---'))

Using


-g4 + sqrt(6)*rho22

to solve for


Matrix([[rho22]])

Solution:


...

Matrix([[g0 - sqrt(2)*g2/6 - sqrt(6)*g4/18 - 2*t2]])

Resulting in


Matrix([
[g0 - sqrt(2)*g2/6 - sqrt(6)*g4/18 - 2*t2,                g1/2 - sqrt(6)*g3/6, g2/3 - 2*sqrt(3)*g4/9 - sqrt(2)*t2],
[                     g1/2 - sqrt(6)*g3/6, sqrt(2)*g2/6 - sqrt(6)*g4/9 + 2*t2,                       sqrt(3)*g3/6],
[      g2/3 - 2*sqrt(3)*g4/9 - sqrt(2)*t2,                       sqrt(3)*g3/6,                       sqrt(6)*g4/6]])

Matrix([
[                                                0],
[                                                0],
[                                                0],
[                                                0],
[                                                0],
[-T + g0/4 - sqrt(2)*g2/12 + 2*sqrt(6)*g4/9 + 2*t2]])

---

In [7]:
#print(rho._repr_latex_())
rho

Matrix([
[g0 - sqrt(2)*g2/6 - sqrt(6)*g4/18 - 2*t2,                g1/2 - sqrt(6)*g3/6, g2/3 - 2*sqrt(3)*g4/9 - sqrt(2)*t2],
[                     g1/2 - sqrt(6)*g3/6, sqrt(2)*g2/6 - sqrt(6)*g4/9 + 2*t2,                       sqrt(3)*g3/6],
[      g2/3 - 2*sqrt(3)*g4/9 - sqrt(2)*t2,                       sqrt(3)*g3/6,                       sqrt(6)*g4/6]])

In [8]:
#print(equations[-1]._repr_latex_())
equations[-1]

-T + g0/4 - sqrt(2)*g2/12 + 2*sqrt(6)*g4/9 + 2*t2

## Free parameter

In [9]:
from sympy.abc import T

t2 = Symbol('t2')

In [10]:
from sympy.abc import lamda

P_char = Poly((lamda**(D-1)*(lamda-1)**2).expand(), lamda)
P_char

Poly(lamda**3 - 2*lamda**2 + lamda, lamda, domain='ZZ')

In [11]:
P_rho = rho.charpoly(lamda)
P_rho

PurePoly(lamda**3 - g0*lamda**2 + (sqrt(2)*g0*g2/6 + sqrt(6)*g0*g4/18 + 2*g0*t2 - g1**2/4 + sqrt(6)*g1*g3/6 - g2**2/6 + sqrt(3)*g2*g4/6 - g3**2/4 - 5*g4**2/18 - sqrt(6)*g4*t2/3 - 6*t2**2)*lamda - sqrt(3)*g0*g2*g4/18 + g0*g3**2/12 + g0*g4**2/9 - sqrt(6)*g0*g4*t2/3 + sqrt(6)*g1**2*g4/24 - sqrt(3)*g1*g2*g3/18 - g1*g3*g4/18 + sqrt(6)*g1*g3*t2/6 + sqrt(2)*g2**3/54 - sqrt(6)*g2**2*g4/36 + sqrt(2)*g2*g3**2/24 + 7*sqrt(2)*g2*g4**2/108 + 2*sqrt(3)*g2*g4*t2/9 - sqrt(2)*g2*t2**2 - sqrt(6)*g3**2*g4/72 - g3**2*t2/2 - 11*sqrt(6)*g4**3/486 - g4**2*t2/9 + 4*sqrt(6)*g4*t2**2/3 + 4*t2**3, lamda, domain='EX')

In [12]:
equations2 = [Poly(c, t2) for c in (P_rho-P_char).as_list()]
for eq in equations2:
    display(eq)

Poly(2 - g0, t2, domain='ZZ[g0]')

Poly(-6*t2**2 + (2*g0 - sqrt(6)*g4/3)*t2 + sqrt(2)*g0*g2/6 + sqrt(6)*g0*g4/18 - g1**2/4 + sqrt(6)*g1*g3/6 - g2**2/6 + sqrt(3)*g2*g4/6 - g3**2/4 - 5*g4**2/18 - 1, t2, domain='EX')

Poly(4*t2**3 + (-sqrt(2)*g2 + 4*sqrt(6)*g4/3)*t2**2 + (-sqrt(6)*g0*g4/3 + sqrt(6)*g1*g3/6 + 2*sqrt(3)*g2*g4/9 - g3**2/2 - g4**2/9)*t2 - sqrt(3)*g0*g2*g4/18 + g0*g3**2/12 + g0*g4**2/9 + sqrt(6)*g1**2*g4/24 - sqrt(3)*g1*g2*g3/18 - g1*g3*g4/18 + sqrt(2)*g2**3/54 - sqrt(6)*g2**2*g4/36 + sqrt(2)*g2*g3**2/24 + 7*sqrt(2)*g2*g4**2/108 - sqrt(6)*g3**2*g4/72 - 11*sqrt(6)*g4**3/486, t2, domain='EX')

In [13]:
subs = {Symbol('g0'):2}
rho = rho.subs(subs)
rho

Matrix([
[-sqrt(2)*g2/6 - sqrt(6)*g4/18 - 2*t2 + 2,                g1/2 - sqrt(6)*g3/6, g2/3 - 2*sqrt(3)*g4/9 - sqrt(2)*t2],
[                     g1/2 - sqrt(6)*g3/6, sqrt(2)*g2/6 - sqrt(6)*g4/9 + 2*t2,                       sqrt(3)*g3/6],
[      g2/3 - 2*sqrt(3)*g4/9 - sqrt(2)*t2,                       sqrt(3)*g3/6,                       sqrt(6)*g4/6]])

In [14]:
equations[-1] = equations[-1].subs(subs)
equations[-1]

-T - sqrt(2)*g2/12 + 2*sqrt(6)*g4/9 + 2*t2 + 1/2

### Idempotency

In [15]:
M = (rho**2 - rho).expand()
equations2 = []
for i in range(rho.shape[0]):
    for j in range(rho.shape[1]):
        eq = Poly(M[i,j], t2)
        #don't take duplicates
        if not any(eq-e==0 for e in equations2):
            equations2.append(eq)

for eq in equations2:
    #print(eq._repr_latex_())
    display(eq)

Poly(6*t2**2 + (2*sqrt(6)*g4/3 - 6)*t2 + g1**2/4 - sqrt(6)*g1*g3/6 + g2**2/6 - sqrt(3)*g2*g4/9 - sqrt(2)*g2/2 + g3**2/6 + g4**2/6 - sqrt(6)*g4/6 + 2, t2, domain='EX')

Poly(-sqrt(6)*g3/6*t2 - sqrt(6)*g1*g4/12 + g1/2 + sqrt(3)*g2*g3/18 + g3*g4/18 - sqrt(6)*g3/6, t2, domain='EX')

Poly(2*sqrt(2)*t2**2 + (-g2/3 + 2*sqrt(3)*g4/9 - sqrt(2))*t2 + sqrt(3)*g1*g3/12 - sqrt(2)*g2**2/18 + 2*sqrt(6)*g2*g4/27 + g2/3 - sqrt(2)*g3**2/12 - 2*sqrt(2)*g4**2/27 - 2*sqrt(3)*g4/9, t2, domain='EX')

Poly(4*t2**2 + (2*sqrt(2)*g2/3 - 4*sqrt(6)*g4/9 - 2)*t2 + g1**2/4 - sqrt(6)*g1*g3/6 + g2**2/18 - 2*sqrt(3)*g2*g4/27 - sqrt(2)*g2/6 + g3**2/4 + 2*g4**2/27 + sqrt(6)*g4/9, t2, domain='EX')

Poly((-sqrt(2)*g1/2 + 2*sqrt(3)*g3/3)*t2 + g1*g2/6 - sqrt(3)*g1*g4/9 - sqrt(6)*g2*g3/36 + 5*sqrt(2)*g3*g4/36 - sqrt(3)*g3/6, t2, domain='EX')

Poly(2*t2**2 + (-2*sqrt(2)*g2/3 + 4*sqrt(6)*g4/9)*t2 + g2**2/9 - 4*sqrt(3)*g2*g4/27 + g3**2/12 + 17*g4**2/54 - sqrt(6)*g4/6, t2, domain='EX')

## Linear combination

In [16]:
from math import sumprod

a = Matrix(symbols(f'a:{len(equations2)}'))
a_sol = Matrix(a)
P = Poly(sumprod(a, equations2), t2)

def sol(idx, sym_idx):
    global P, a, a_sol
    
    print('Using')
    eq = Poly(P.as_dict()[idx], a[sym_idx])
    display(eq)
    print('to solve for', a[sym_idx], '...')
    
    a_sol[sym_idx] = eq.root(0)
    print('Found solution')
    display(a_sol[sym_idx])
    
    subs = {a[sym_idx]: a_sol[sym_idx]}
    for i in range(len(a_sol)):
        a_sol[i] = Poly(a_sol[i].subs(subs), *a).as_expr()
    display(a_sol)
    
    P = Poly(P.subs(subs), t2)
    display(P)

P

Poly((6*a0 + 2*sqrt(2)*a2 + 4*a3 + 2*a5)*t2**2 + (2*sqrt(6)*a0*g4/3 - 6*a0 - sqrt(6)*a1*g3/6 - a2*g2/3 + 2*sqrt(3)*a2*g4/9 - sqrt(2)*a2 + 2*sqrt(2)*a3*g2/3 - 4*sqrt(6)*a3*g4/9 - 2*a3 - sqrt(2)*a4*g1/2 + 2*sqrt(3)*a4*g3/3 - 2*sqrt(2)*a5*g2/3 + 4*sqrt(6)*a5*g4/9)*t2 + a0*g1**2/4 - sqrt(6)*a0*g1*g3/6 + a0*g2**2/6 - sqrt(3)*a0*g2*g4/9 - sqrt(2)*a0*g2/2 + a0*g3**2/6 + a0*g4**2/6 - sqrt(6)*a0*g4/6 + 2*a0 - sqrt(6)*a1*g1*g4/12 + a1*g1/2 + sqrt(3)*a1*g2*g3/18 + a1*g3*g4/18 - sqrt(6)*a1*g3/6 + sqrt(3)*a2*g1*g3/12 - sqrt(2)*a2*g2**2/18 + 2*sqrt(6)*a2*g2*g4/27 + a2*g2/3 - sqrt(2)*a2*g3**2/12 - 2*sqrt(2)*a2*g4**2/27 - 2*sqrt(3)*a2*g4/9 + a3*g1**2/4 - sqrt(6)*a3*g1*g3/6 + a3*g2**2/18 - 2*sqrt(3)*a3*g2*g4/27 - sqrt(2)*a3*g2/6 + a3*g3**2/4 + 2*a3*g4**2/27 + sqrt(6)*a3*g4/9 + a4*g1*g2/6 - sqrt(3)*a4*g1*g4/9 - sqrt(6)*a4*g2*g3/36 + 5*sqrt(2)*a4*g3*g4/36 - sqrt(3)*a4*g3/6 + a5*g2**2/9 - 4*sqrt(3)*a5*g2*g4/27 + a5*g3**2/12 + 17*a5*g4**2/54 - sqrt(6)*a5*g4/6, t2, domain='EX')

In [17]:
sol((2,), 5)

Using


Poly(2*a5 + 6*a0 + 2*sqrt(2)*a2 + 4*a3, a5, domain='EX')

to solve for a5 ...
Found solution


-3*a0 - sqrt(2)*a2 - 2*a3

Matrix([
[                       a0],
[                       a1],
[                       a2],
[                       a3],
[                       a4],
[-3*a0 - sqrt(2)*a2 - 2*a3]])

Poly((2*sqrt(2)*a0*g2 - 2*sqrt(6)*a0*g4/3 - 6*a0 - sqrt(6)*a1*g3/6 + a2*g2 - 2*sqrt(3)*a2*g4/3 - sqrt(2)*a2 + 2*sqrt(2)*a3*g2 - 4*sqrt(6)*a3*g4/3 - 2*a3 - sqrt(2)*a4*g1/2 + 2*sqrt(3)*a4*g3/3)*t2 + a0*g1**2/4 - sqrt(6)*a0*g1*g3/6 - a0*g2**2/6 + sqrt(3)*a0*g2*g4/3 - sqrt(2)*a0*g2/2 - a0*g3**2/12 - 7*a0*g4**2/9 + sqrt(6)*a0*g4/3 + 2*a0 - sqrt(6)*a1*g1*g4/12 + a1*g1/2 + sqrt(3)*a1*g2*g3/18 + a1*g3*g4/18 - sqrt(6)*a1*g3/6 + sqrt(3)*a2*g1*g3/12 - sqrt(2)*a2*g2**2/6 + 2*sqrt(6)*a2*g2*g4/9 + a2*g2/3 - sqrt(2)*a2*g3**2/6 - 7*sqrt(2)*a2*g4**2/18 + sqrt(3)*a2*g4/9 + a3*g1**2/4 - sqrt(6)*a3*g1*g3/6 - a3*g2**2/6 + 2*sqrt(3)*a3*g2*g4/9 - sqrt(2)*a3*g2/6 + a3*g3**2/12 - 5*a3*g4**2/9 + 4*sqrt(6)*a3*g4/9 + a4*g1*g2/6 - sqrt(3)*a4*g1*g4/9 - sqrt(6)*a4*g2*g3/36 + 5*sqrt(2)*a4*g3*g4/36 - sqrt(3)*a4*g3/6, t2, domain='EX')

In [18]:
Q = Poly(P.as_dict()[1,], *symbols(f'g:{len(g)}'))
Q

Poly(-sqrt(2)*a4/2*g1 + (2*sqrt(2)*a0 + a2 + 2*sqrt(2)*a3)*g2 + (-sqrt(6)*a1/6 + 2*sqrt(3)*a4/3)*g3 + (-2*sqrt(6)*a0/3 - 2*sqrt(3)*a2/3 - 4*sqrt(6)*a3/3)*g4 - 6*a0 - sqrt(2)*a2 - 2*a3, g0, g1, g2, g3, g4, domain='EX')

In [19]:
def sol2(idx, sym_idx):
    global P, Q, a, a_sol
    
    print('Using')
    eq = Poly(Q.as_dict()[idx], a[sym_idx])
    display(eq)
    print('to solve for', a[sym_idx], '...')
    
    a_sol[sym_idx] = eq.root(0)
    print('Found solution')
    display(a_sol[sym_idx])
    
    subs = {a[sym_idx]: a_sol[sym_idx]}
    for i in range(len(a_sol)):
        a_sol[i] = Poly(a_sol[i].subs(subs), *a).as_expr()
    display(a_sol)
    
    P = Poly(P.subs(subs), t2)
    display(P)
    Q = Poly(Q.subs(subs), *symbols('g:5'))
    display(Q)

sol2((0,1,0,0,0), 4)

Using


Poly(-sqrt(2)/2*a4, a4, domain='EX')

to solve for a4 ...
Found solution


0

Matrix([
[                       a0],
[                       a1],
[                       a2],
[                       a3],
[                        0],
[-3*a0 - sqrt(2)*a2 - 2*a3]])

Poly((2*sqrt(2)*a0*g2 - 2*sqrt(6)*a0*g4/3 - 6*a0 - sqrt(6)*a1*g3/6 + a2*g2 - 2*sqrt(3)*a2*g4/3 - sqrt(2)*a2 + 2*sqrt(2)*a3*g2 - 4*sqrt(6)*a3*g4/3 - 2*a3)*t2 + a0*g1**2/4 - sqrt(6)*a0*g1*g3/6 - a0*g2**2/6 + sqrt(3)*a0*g2*g4/3 - sqrt(2)*a0*g2/2 - a0*g3**2/12 - 7*a0*g4**2/9 + sqrt(6)*a0*g4/3 + 2*a0 - sqrt(6)*a1*g1*g4/12 + a1*g1/2 + sqrt(3)*a1*g2*g3/18 + a1*g3*g4/18 - sqrt(6)*a1*g3/6 + sqrt(3)*a2*g1*g3/12 - sqrt(2)*a2*g2**2/6 + 2*sqrt(6)*a2*g2*g4/9 + a2*g2/3 - sqrt(2)*a2*g3**2/6 - 7*sqrt(2)*a2*g4**2/18 + sqrt(3)*a2*g4/9 + a3*g1**2/4 - sqrt(6)*a3*g1*g3/6 - a3*g2**2/6 + 2*sqrt(3)*a3*g2*g4/9 - sqrt(2)*a3*g2/6 + a3*g3**2/12 - 5*a3*g4**2/9 + 4*sqrt(6)*a3*g4/9, t2, domain='EX')

Poly((2*sqrt(2)*a0 + a2 + 2*sqrt(2)*a3)*g2 - sqrt(6)*a1/6*g3 + (-2*sqrt(6)*a0/3 - 2*sqrt(3)*a2/3 - 4*sqrt(6)*a3/3)*g4 - 6*a0 - sqrt(2)*a2 - 2*a3, g0, g1, g2, g3, g4, domain='EX')

In [20]:
sol2((0,0,0,1,0), 1)

Using


Poly(-sqrt(6)/6*a1, a1, domain='EX')

to solve for a1 ...
Found solution


0

Matrix([
[                       a0],
[                        0],
[                       a2],
[                       a3],
[                        0],
[-3*a0 - sqrt(2)*a2 - 2*a3]])

Poly((2*sqrt(2)*a0*g2 - 2*sqrt(6)*a0*g4/3 - 6*a0 + a2*g2 - 2*sqrt(3)*a2*g4/3 - sqrt(2)*a2 + 2*sqrt(2)*a3*g2 - 4*sqrt(6)*a3*g4/3 - 2*a3)*t2 + a0*g1**2/4 - sqrt(6)*a0*g1*g3/6 - a0*g2**2/6 + sqrt(3)*a0*g2*g4/3 - sqrt(2)*a0*g2/2 - a0*g3**2/12 - 7*a0*g4**2/9 + sqrt(6)*a0*g4/3 + 2*a0 + sqrt(3)*a2*g1*g3/12 - sqrt(2)*a2*g2**2/6 + 2*sqrt(6)*a2*g2*g4/9 + a2*g2/3 - sqrt(2)*a2*g3**2/6 - 7*sqrt(2)*a2*g4**2/18 + sqrt(3)*a2*g4/9 + a3*g1**2/4 - sqrt(6)*a3*g1*g3/6 - a3*g2**2/6 + 2*sqrt(3)*a3*g2*g4/9 - sqrt(2)*a3*g2/6 + a3*g3**2/12 - 5*a3*g4**2/9 + 4*sqrt(6)*a3*g4/9, t2, domain='EX')

Poly((2*sqrt(2)*a0 + a2 + 2*sqrt(2)*a3)*g2 + (-2*sqrt(6)*a0/3 - 2*sqrt(3)*a2/3 - 4*sqrt(6)*a3/3)*g4 - 6*a0 - sqrt(2)*a2 - 2*a3, g0, g1, g2, g3, g4, domain='EX')

In [21]:
sol2((0,0,1,0,0), 3)

Using


Poly(2*sqrt(2)*a3 + 2*sqrt(2)*a0 + a2, a3, domain='EX')

to solve for a3 ...
Found solution


-a0 - sqrt(2)*a2/4

Matrix([
[                a0],
[                 0],
[                a2],
[-a0 - sqrt(2)*a2/4],
[                 0],
[-a0 - sqrt(2)*a2/2]])

Poly((2*sqrt(6)*a0*g4/3 - 4*a0 - sqrt(2)*a2/2)*t2 + sqrt(3)*a0*g2*g4/9 - sqrt(2)*a0*g2/3 - a0*g3**2/6 - 2*a0*g4**2/9 - sqrt(6)*a0*g4/9 + 2*a0 - sqrt(2)*a2*g1**2/16 + sqrt(3)*a2*g1*g3/6 - sqrt(2)*a2*g2**2/8 + sqrt(6)*a2*g2*g4/6 + 5*a2*g2/12 - 3*sqrt(2)*a2*g3**2/16 - sqrt(2)*a2*g4**2/4 - sqrt(3)*a2*g4/9, t2, domain='EX')

Poly(2*sqrt(6)*a0/3*g4 - 4*a0 - sqrt(2)*a2/2, g0, g1, g2, g3, g4, domain='EX')

In [22]:
sol2((0,0,0,0,1), 0)

Using


Poly(2*sqrt(6)/3*a0, a0, domain='EX')

to solve for a0 ...
Found solution


0

Matrix([
[            0],
[            0],
[           a2],
[-sqrt(2)*a2/4],
[            0],
[-sqrt(2)*a2/2]])

Poly(-sqrt(2)*a2/2*t2 - sqrt(2)*a2*g1**2/16 + sqrt(3)*a2*g1*g3/6 - sqrt(2)*a2*g2**2/8 + sqrt(6)*a2*g2*g4/6 + 5*a2*g2/12 - 3*sqrt(2)*a2*g3**2/16 - sqrt(2)*a2*g4**2/4 - sqrt(3)*a2*g4/9, t2, domain='EX')

Poly(-sqrt(2)*a2/2, g0, g1, g2, g3, g4, domain='EX')

In [23]:
a_sol = a_sol.subs({a[2]:sqrt(2)})
a_sol

Matrix([
[      0],
[      0],
[sqrt(2)],
[   -1/2],
[      0],
[     -1]])

In [24]:
subs = {ai:a_soli for ai, a_soli in zip(a, a_sol)}

P = Poly(P.subs(subs), t2)
#print(P._repr_latex_())
P

Poly(-t2 - g1**2/8 + sqrt(6)*g1*g3/6 - g2**2/4 + sqrt(3)*g2*g4/3 + 5*sqrt(2)*g2/12 - 3*g3**2/8 - g4**2/2 - sqrt(6)*g4/9, t2, domain='EX')

In [25]:
t2_sol = P.root(0)
t2_sol

-g1**2/8 + sqrt(6)*g1*g3/6 - g2**2/4 + sqrt(3)*g2*g4/3 + 5*sqrt(2)*g2/12 - 3*g3**2/8 - g4**2/2 - sqrt(6)*g4/9

In [26]:
subs = {t2:t2_sol}
rho = rho.subs(subs).expand()
#print(rho._repr_latex_())
rho

Matrix([
[       g1**2/4 - sqrt(6)*g1*g3/3 + g2**2/2 - 2*sqrt(3)*g2*g4/3 - sqrt(2)*g2 + 3*g3**2/4 + g4**2 + sqrt(6)*g4/6 + 2,                                                                                      g1/2 - sqrt(6)*g3/6, sqrt(2)*g1**2/8 - sqrt(3)*g1*g3/3 + sqrt(2)*g2**2/4 - sqrt(6)*g2*g4/3 - g2/2 + 3*sqrt(2)*g3**2/8 + sqrt(2)*g4**2/2],
[                                                                                               g1/2 - sqrt(6)*g3/6, -g1**2/4 + sqrt(6)*g1*g3/3 - g2**2/2 + 2*sqrt(3)*g2*g4/3 + sqrt(2)*g2 - 3*g3**2/4 - g4**2 - sqrt(6)*g4/3,                                                                                                       sqrt(3)*g3/6],
[sqrt(2)*g1**2/8 - sqrt(3)*g1*g3/3 + sqrt(2)*g2**2/4 - sqrt(6)*g2*g4/3 - g2/2 + 3*sqrt(2)*g3**2/8 + sqrt(2)*g4**2/2,                                                                                             sqrt(3)*g3/6,                                                                                         

In [27]:
equations[-1] = equations[-1].subs(subs).expand()
#print(equations[-1]._repr_latex_())
equations[-1]

-T - g1**2/4 + sqrt(6)*g1*g3/3 - g2**2/2 + 2*sqrt(3)*g2*g4/3 + 3*sqrt(2)*g2/4 - 3*g3**2/4 - g4**2 + 1/2

In [28]:
T = Poly(equations[-1], T).root(0)
#print(T._repr_latex_())
T

-g1**2/4 + sqrt(6)*g1*g3/3 - g2**2/2 + 2*sqrt(3)*g2*g4/3 + 3*sqrt(2)*g2/4 - 3*g3**2/4 - g4**2 + 1/2

---

In [29]:
T_matrix = T_matrix(D)
for _ in range(10):
    v, w = rand_ortho_pair(D+1)
    rho = v*v.T + w*w.T
    g = rho_to_g(rho)
    T_predicted = T.subs({Symbol(f'g{i}'):g[i] for i in range(len(g))}).simplify()
    T_actually = (rho @ T_matrix).trace().simplify()
    assert (T_predicted - T_actually).simplify().equals(0)